# Round 3 notebook 03 — option surface and bot behavior

        This notebook reverse-engineers the voucher family as a market generated by:
        a delta-1 underlying, deep-ITM parity vouchers, a near-money option surface, and persistent OTM seller bots.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw))
trades = attach_trades(prices, trades_raw)

display(Markdown("## Voucher side asymmetry"))
side = pd.crosstab(trades["symbol"], trades["side"])
side["total"] = side.sum(axis=1)
side["sell_agg_frac"] = (side.get("sell_agg", 0) / side["total"]).round(3)
display(side.sort_index())

display(Markdown("## Maker markout: what bot flow is worth if we are resting at the touched level"))
mk = maker_markout_table(trades)
display(mk[mk["symbol"].str.startswith("VEV") & (mk["n"] >= 5)].sort_values(["symbol", "side"]))

display(Markdown("## Delta-1 parity checks"))
wide = prices.pivot_table(index=["day", "timestamp"], columns="product", values="mid_price")
rows = []
for k in [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]:
    sym = f"VEV_{k}"
    basis = wide[sym] - np.maximum(wide["VELVETFRUIT_EXTRACT"] - k, 0)
    rows.append({
        "strike": k,
        "basis_mean": basis.mean(),
        "basis_std": basis.std(),
        "basis_min": basis.min(),
        "basis_max": basis.max(),
        "corr_underlying": wide[sym].corr(wide["VELVETFRUIT_EXTRACT"]),
    })
display(pd.DataFrame(rows).round(4))

display(Markdown("## Slow-anchor payoff distance across the surface"))
display(slow_ema_payoff(prices, span=1000))

display(Markdown("## OTM seller fingerprint"))
otm = trades[trades["symbol"].isin(["VEV_5200", "VEV_5300", "VEV_5400", "VEV_5500", "VEV_6000", "VEV_6500"])].copy()
otm["strike"] = otm["symbol"].str.extract(r"(\d+)").astype(int)
display(otm.groupby(["symbol", "side"]).agg(
    n=("price", "size"),
    avg_qty=("quantity", "mean"),
    avg_price=("price", "mean"),
    first_ts=("timestamp", "min"),
    last_ts=("timestamp", "max"),
).round(3))

## Voucher side asymmetry

side,buy_agg,other,sell_agg,total,sell_agg_frac
symbol,,,,,
HYDROGEL_PACK,524,0,486,1010,0.481
VELVETFRUIT_EXTRACT,777,6,589,1372,0.429
VEV_4000,226,0,238,464,0.513
VEV_4500,1,0,0,1,0.000
VEV_5000,1,0,0,1,0.000
VEV_5100,1,0,0,1,0.000
VEV_5200,1,0,17,18,0.944
VEV_5300,1,1,119,121,0.983
VEV_5400,0,0,225,225,1.000


## Maker markout: what bot flow is worth if we are resting at the touched level

,symbol,side,n,avg_qty,avg_price,mk_1,mk_10,mk_50,mk_100,mk_200,mk_500,mk_1000,mk_2000,mk_5000
5,VEV_4000,buy_agg,226,2.013,1259.695,10.409,10.089,10.400,10.207,10.868,10.384,9.814,7.702,11.068
6,VEV_4000,sell_agg,238,2.038,1240.433,10.515,10.305,10.141,9.722,9.919,10.581,10.482,10.594,8.016
11,VEV_5200,sell_agg,17,3.647,85.176,1.206,1.000,1.618,1.000,1.688,3.906,8.800,9.357,20.857
14,VEV_5300,sell_agg,119,3.479,44.235,0.765,0.706,0.643,0.829,0.978,0.942,2.722,3.202,-0.560
15,VEV_5400,sell_agg,225,3.498,14.884,0.573,0.547,0.593,0.489,0.448,0.616,0.745,0.453,-1.555
16,VEV_5500,sell_agg,267,3.509,5.951,0.517,0.519,0.556,0.519,0.502,0.525,0.612,0.523,-0.485
17,VEV_6000,sell_agg,284,3.528,0.000,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.500
18,VEV_6500,sell_agg,284,3.528,0.000,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.500,0.500


## Delta-1 parity checks

...

,strike,basis_mean,basis_std,basis_min,basis_max,corr_underlying
0,4000,0.0117,0.8287,-7.0,7.0,0.9986
1,4500,0.0115,0.7589,-6.0,5.5,0.9988
2,5000,4.9243,1.9298,-0.5,11.5,0.9952
3,5100,16.7074,4.8425,4.5,29.5,0.9621
4,5200,45.4504,7.8466,21.5,67.5,0.9140
5,5300,46.7599,6.2281,26.5,65.5,0.8372
6,5400,15.9519,3.4292,6.5,27.0,0.5732
7,5500,6.6414,1.7388,2.5,12.5,0.4933
8,6000,0.5000,0.0000,0.5,0.5,NaN
9,6500,0.5000,0.0000,0.5,0.5,NaN


## Slow-anchor payoff distance across the surface

,product,dev_q10,dev_q90,best_H,edge_ticks,low_future_move,high_future_move
0,HYDROGEL_PACK,-25.577,26.477,2000.0,32.039,35.430,-28.648
2,VEV_4000,-13.900,14.648,1000.0,20.881,23.099,-18.664
3,VEV_4500,-13.846,14.633,1000.0,20.879,23.058,-18.700
1,VELVETFRUIT_EXTRACT,-13.886,14.654,1000.0,20.770,22.864,-18.676
4,VEV_5000,-12.984,13.598,1000.0,19.531,21.280,-17.782
5,VEV_5100,-11.534,11.997,1000.0,16.831,17.917,-15.745
6,VEV_5200,-8.717,8.890,1000.0,12.638,12.839,-12.436
7,VEV_5300,-5.499,5.599,1000.0,8.101,7.792,-8.410
8,VEV_5400,-2.684,2.536,1000.0,3.497,2.607,-4.387
9,VEV_5500,-1.213,1.146,1000.0,1.595,1.174,-2.017


## OTM seller fingerprint

n  avg_qty  avg_price  first_ts  last_ts
symbol   side                                                
VEV_5200 buy_agg     1    1.000     87.000    437100   437100
         sell_agg   17    3.647     85.176    220100   988100
VEV_5300 buy_agg     1    1.000     41.000    437100   437100
         other       1    5.000     38.000    554600   554600
         sell_agg  119    3.479     44.235     14800   994300
VEV_5400 sell_agg  225    3.498     14.884      2900   994300
VEV_5500 sell_agg  267    3.509      5.951      2900   994300
VEV_6000 sell_agg  284    3.528      0.000      2900   994300
VEV_6500 sell_agg  284    3.528      0.000      2900   994300

## Option-bot interpretation

        - The deep-ITM calls are near-perfect delta-1 mirrors. `VEV_4000` and `VEV_4500` should be treated as underlying-like spread products, not as normal options.
        - The active OTM vouchers have a persistent seller hitting bids. That is the clearest bot behavior in R3.
        - However, for `VEV_6000/6500`, the visible mark-to-mid is a trap: the market is pinned at `0/1`, but prior backtests showed bid-0 accumulation has zero realized EV.
        - For `VEV_5300/5400/5500`, the bot seller creates a long-biased optionality accumulation edge, but execution must pay attention to whether fills are possible in `none` mode versus website/passive modes.
        - The slow-anchor David sleeve fits the data because each active voucher mean-reverts around a long-run reference; the surface is less important than the hidden taker/maker ecology.